# REINFORCE (PyTorch Scaffolding)

Implement the vanilla policy-gradient algorithm (REINFORCE) in PyTorch.

Focus areas:
- categorical policy parameterization
- trajectory rollout
- discounted return computation
- policy-gradient loss and optimization

## 1) Setup

In [1]:
import math
from dataclasses import dataclass
from typing import List, Tuple

import torch
import torch.nn as nn
from torch.distributions import Categorical

torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## 2) Tiny environment (bandit-style MDP)

This keeps the scaffold self-contained and easy to debug.
State is a one-hot context; reward depends on action and context.

In [3]:
@dataclass
class TinyMDP:
    state_dim: int = 4
    action_dim: int = 2 # only two possible actions
    horizon: int = 12

    def reset(self):
        self.t = 0
        self.s = torch.randint(low=0, high=self.state_dim, size=(1,)).item() # randomly initialized state space
        return self._obs(self.s) # the current state

    def _obs(self, s_idx: int): # observation is one-hot encoding
        x = torch.zeros(self.state_dim, dtype=torch.float32)
        x[s_idx] = 1.0
        return x

    def step(self, action: int):
        # Reward rule: contexts 0/1 prefer action 0; contexts 2/3 prefer action 1
        good_action = 0 if self.s < 2 else 1
        reward = 1.0 if action == good_action else -0.2

        self.t += 1
        done = self.t >= self.horizon

        # Transition to a new random context
        self.s = torch.randint(low=0, high=self.state_dim, size=(1,)).item()
        next_state = self._obs(self.s)
        return next_state, reward, done, {}

## 3) Policy network

In [4]:
class PolicyNet(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 32):
        super().__init__() # initialize the neural network
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.Tanh(), # why tanh activation?
            nn.Linear(hidden, action_dim),
        )

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        # TODO: return action logits of shape [action_dim] for single state
        # probabiltiies over actions
        # raise NotImplementedError

        return self.net(state)

    def dist(self, state: torch.Tensor) -> Categorical:
        # TODO: construct categorical distribution from logits
        # raise NotImplementedError

        logits = self.forward(state)
        return Categorical(logits=logits)

## 4) Rollout collection

Collect one episode and store tensors needed for REINFORCE:
- log-probabilities of sampled actions
- rewards

In [12]:
def collect_episode(env: TinyMDP, policy: PolicyNet):
    # TODO:
    # 1) reset env
    # 2) loop until done:
    #    - sample action from policy.dist(state)
    #    - step env
    #    - store log_prob(action), reward
    # 3) return log_probs (list of tensors), rewards (list of floats)
    # raise NotImplementedError

    obs = env.reset()
    done = False
    log_probs = []
    rewards = []

    while not done:
        log_prob = policy.dist(obs)
        action = log_prob.sample()
        next_obs, reward, done, info = env.step(action)

        rewards.append(reward)
        log_probs.append(log_prob.probs[action])

        obs = next_obs

    return log_probs, rewards

## 5) Discounted returns

In [18]:
def discounted_returns(rewards: List[float], gamma: float = 0.99) -> torch.Tensor:
    # TODO: compute G_t = r_t + gamma * G_{t+1}, backward pass
    # return 1D tensor [T]
    # raise NotImplementedError

    H = len(rewards)
    returns = torch.empty(H)
    returns[H - 1] = rewards[H - 1]

    for i in range(H - 2, -1, -1):
        returns[i] = rewards[i] + gamma * returns[i + 1] # can implement with running to prevent index look up
    return returns

# Optional: normalize returns for variance reduction
def normalize(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    return (x - x.mean()) / (x.std(unbiased=False) + eps)

## 6) REINFORCE loss

Implement policy loss:
`L = -sum_t log pi(a_t|s_t) * G_t`

You may average over time steps instead of summing.

In [9]:
def reinforce_loss(log_probs: List[torch.Tensor], returns: torch.Tensor):
    # TODO:
    # stacked = torch.stack(log_probs)
    # loss = -(stacked * returns).sum()   # or .mean()
    # return loss
    # raise NotImplementedError

    stacked = torch.stack(log_probs)
    loss = -(stacked * returns).sum()
    return loss

## 7) Training loop

In [14]:
def train_reinforce(episodes: int = 300, gamma: float = 0.99, lr: float = 1e-2):
    env = TinyMDP()
    policy = PolicyNet(env.state_dim, env.action_dim).to(device)
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)

    episode_returns = []

    for ep in range(episodes):
        # TODO:
        log_probs, rewards = collect_episode(env, policy)
        returns = discounted_returns(rewards, gamma)
        returns = normalize(returns)
        loss = reinforce_loss(log_probs, returns)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        episode_returns.append(sum(rewards))
        # raise NotImplementedError

        if (ep + 1) % 50 == 0:
            avg = sum(episode_returns[-50:]) / 50
            print(f"episode {ep+1:4d} | avg_return(last50)={avg:.3f}")

    return policy, episode_returns

## 8) Smoke checks (uncomment after TODOs)

In [19]:
env = TinyMDP()
policy = PolicyNet(env.state_dim, env.action_dim)

s = env.reset()
d = policy.dist(s)
a = d.sample()
print("sampled action:", int(a))

log_probs, rewards = collect_episode(env, policy)
rets = discounted_returns(rewards, gamma=0.99)
print("episode length:", len(rewards), "returns shape:", tuple(rets.shape))

loss = reinforce_loss(log_probs, normalize(rets))
print("loss:", float(loss))

trained_policy, hist = train_reinforce(episodes=300)
print("final avg return (last 20):", sum(hist[-20:]) / 20)

sampled action: 0
episode length: 12 returns shape: (12,)
loss: 0.017060458660125732
episode   50 | avg_return(last50)=6.864
episode  100 | avg_return(last50)=9.168
episode  150 | avg_return(last50)=11.376
episode  200 | avg_return(last50)=11.712
episode  250 | avg_return(last50)=11.808
episode  300 | avg_return(last50)=11.856
final avg return (last 20): 11.879999999999999


## 9) Stretch prompts

- Add a baseline value network and switch to advantage estimates.
- Compare no-normalization vs normalization of returns.
- Track policy entropy and add entropy regularization.
- Replace `TinyMDP` with a Gymnasium environment and adapt tensor shapes.